In [ ]:

import dash
from dash import dcc, html, Input, Output, State
import pandas as pd
import plotly.express as px
import os

# Initialize the Dash app
app = dash.Dash(__name__)
server = app.server  # Required for Render deployment

  Using cached dash-4.1.0-py3-none-any.whl.metadata (11 kB)
  Using cached pandas-3.0.2-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached plotly-6.7.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached flask-3.1.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached importlib_metadata-9.0.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached requests-2.33.1-py3-none-any.whl.metadata (4.8 kB)
  Using cached click-8.3.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
Using cached dash-4.1.0-py3-none-any.whl (7.2 MB)
Using cached flask-3.1.3-py3-none-any.whl (103 kB)
Using cached pandas-3.0.2-cp314-cp314-win_amd64.whl (9.9 MB)
Using cached plotly-6.7.0-py3-none-any.whl (9.9 MB)
Using cached click-8.3.2-py3-none-any.whl (108 kB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
Using cached importlib_metadata-9.0.0-py3-none-any.whl (27 kB)
Using cached requests-2.33.1-py3-none-any.whl (64 kB)

   ----------------------------------------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Modern Dark Theme Styling
app.layout = html.Div(style={
    'backgroundColor': '#121212', 
    'color': '#e0e0e0', 
    'padding': '40px', 
    'minHeight': '100vh', 
    'fontFamily': '"Segoe UI", Roboto, Helvetica, Arial, sans-serif'
}, children=[
    # Branding Header
    html.Div(style={'textAlign': 'center', 'marginBottom': '40px'}, children=[
        html.H1("Credit Risk Intelligence Portal", 
                style={'color': '#FFD700', 'fontSize': '36px', 'fontWeight': '700', 'letterSpacing': '2px'}),
        html.P("South African Financial Assessment Tool | ZAR", 
                style={'color': '#888', 'fontSize': '16px'})
    ]),

    html.Div(style={'display': 'flex', 'flexWrap': 'wrap', 'justifyContent': 'center', 'gap': '30px'}, children=[
        # Input Section (The "Card")
        html.Div(style={
            'backgroundColor': '#1e1e1e', 
            'padding': '30px', 
            'borderRadius': '15px', 
            'boxShadow': '0 10px 20px rgba(0,0,0,0.4)',
            'width': '400px',
            'borderTop': '4px solid #FFD700'
        }, children=[
            html.H3("Applicant Profile", style={'marginBottom': '20px', 'color': '#ffffff'}),
            
            html.Label("Annual Gross Income (R):", style={'display': 'block', 'marginTop': '10px'}),
            dcc.Input(id='income', type='number', placeholder='e.g., 350000', 
                     style={'width': '100%', 'padding': '12px', 'margin': '8px 0', 'backgroundColor': '#2a2a2a', 'color': '#fff', 'border': '1px solid #444', 'borderRadius': '5px'}),
            
            html.Label("Total Monthly Debt (R):", style={'display': 'block', 'marginTop': '15px'}),
            dcc.Input(id='debt', type='number', placeholder='e.g., 8500', 
                     style={'width': '100%', 'padding': '12px', 'margin': '8px 0', 'backgroundColor': '#2a2a2a', 'color': '#fff', 'border': '1px solid #444', 'borderRadius': '5px'}),
            
            html.Label("Credit Score (0 - 999):", style={'display': 'block', 'marginTop': '15px'}),
            dcc.Input(id='score', type='number', placeholder='Average is ~612', 
                     style={'width': '100%', 'padding': '12px', 'margin': '8px 0', 'backgroundColor': '#2a2a2a', 'color': '#fff', 'border': '1px solid #444', 'borderRadius': '5px'}),

            html.Button('GENERATE RISK ASSESSMENT', id='predict-btn', n_clicks=0, 
                        style={
                            'width': '100%', 'marginTop': '25px', 'padding': '15px', 
                            'backgroundColor': '#FFD700', 'color': '#121212', 
                            'fontWeight': '800', 'border': 'none', 'borderRadius': '5px', 'cursor': 'pointer',
                            'transition': '0.3s'
                        }),
        ]),

        # Display Section (The Results)
        html.Div(style={
            'backgroundColor': '#1e1e1e', 
            'padding': '30px', 
            'borderRadius': '15px', 
            'width': '600px',
            'boxShadow': '0 10px 20px rgba(0,0,0,0.4)'
        }, children=[
            html.Div(id='prediction-output', style={'textAlign': 'center', 'marginBottom': '20px'}),
            dcc.Graph(id='importance-graph', style={'height': '350px'})
        ])
    ])
])

In [3]:
@app.callback(
    [Output('prediction-output', 'children'),
     Output('importance-graph', 'figure')],
    Input('predict-btn', 'n_clicks'),
    [State('income', 'value'), State('debt', 'value'), State('score', 'value')]
)
def update_analysis(n_clicks, income, debt, score):
    # Initialize Graph from P2 artifacts [cite: 8, 9]
    try:
        df_importance = pd.read_csv('artifacts/feature_importance.csv')
        fig = px.bar(df_importance, x='importance', y='feature', orientation='h',
                     template='plotly_dark', color_discrete_sequence=['#FFD700'])
        fig.update_layout(title="Decision Drivers (AI Explainability)", plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)')
    except:
        fig = px.bar(title="Waiting for Model Data (P2)...", template='plotly_dark')

    if n_clicks == 0:
        return html.H4("Awaiting input for assessment...", style={'color': '#666'}), fig

    # Local Risk Logic (SA Banking Standards)
    if score is not None and score < 610:
        res, sub, color = "HIGH RISK", "Low Credit Score Detected", "#FF4C4C"
    elif income and debt and (debt / (income/12) > 0.45):
        res, sub, color = "HIGH RISK", "Debt-to-Income Ratio exceeds 45%", "#FF4C4C"
    else:
        res, sub, color = "LOW RISK", "Approved based on current criteria", "#00FF7F"

    output = html.Div([
        html.H2(res, style={'color': color, 'margin': '0'}),
        html.P(sub, style={'color': '#888', 'marginTop': '5px'})
    ])

    return output, fig

# Command to run the app
if __name__ == '__main__':
    # Use jupyter_mode='inline' if running inside the notebook
    app.run(jupyter_mode='inline')